# Graph Similarity Analysis Notebook

This notebook compares:

1. A **semantic embeddings graph matrix** (distance or similarity), and
2. A **subscriptions graph matrix** (distance or similarity).

It produces:

- **Numerical outputs** (correlation/error/overlap/permutation-test metrics), and
- **Visual outputs** (heatmaps, difference map, edge-weight scatter, distribution overlays, and permutation null plots).

The approach combines multiple perspectives, since no single graph metric captures all structural dimensions.


## Scientific grounding

This notebook operationalizes methods inspired by:

- Mantel, N. (1967), *Cancer Research* — matrix correlation with permutation testing. https://doi.org/10.1158/0008-5472.CAN-27-2-Part-1-209
- Krackhardt, D. (1987), *Social Networks* — QAP-style permutation logic for network association.
- Borgatti, Everett, & Johnson (2018), *Analyzing Social Networks* — interpretation of dyadic matrix comparisons.
- Schieber et al. (2017), *Nature Communications* — motivation for multi-metric network dissimilarity assessment. https://doi.org/10.1038/s41467-017-01825-7

Visualization choices are aligned with standard exploratory data analysis for pairwise matrix comparisons (heatmaps for structure, scatter/distribution views for dyadic relationships, and null-distribution plots for permutation-based inference).


In [9]:
# Core scientific stack for matrix math, statistics, and plotting.
import json
import re
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr


In [10]:
# ----------------------------
# Reusable analysis primitives
# ----------------------------

@dataclass
class SimilarityResult:
    """Container for all numeric outputs from the comparison pipeline."""

    comparison_mode: str
    analysis_space: str
    n_nodes: int
    n_pairs: int
    pearson_r: float
    pearson_p: float
    spearman_rho: float
    spearman_p: float
    cosine_similarity: float
    rmse: float
    mae: float
    edge_jaccard_by_threshold: Dict[str, float]
    mantel_r: float
    mantel_p: float
    qap_r: float
    qap_p: float


def resolve_latest_drive_csv(source: Path, download_dir: Path, label: Optional[str] = None) -> Path:
    """Copy the latest Google Drive CSV artifact to local runtime storage.

    `source` can be either:
    - a direct CSV file path,
    - a directory (the notebook will read `<dir>/adjacency_matrix.csv`), or
    - a glob-style pattern so the latest modified match is selected.
    """
    source = Path(str(source))
    source_str = str(source)

    if source.exists() and source.is_dir():
        candidate = source / 'adjacency_matrix.csv'
        if not candidate.exists():
            raise FileNotFoundError(f"Expected adjacency_matrix.csv under: {source}")
        source = candidate
        source_str = str(source)

    if any(ch in source_str for ch in "*?[]"):
        matches = sorted(Path('/').glob(source_str.lstrip('/')), key=lambda p: p.stat().st_mtime)
        if not matches:
            raise FileNotFoundError(f"No Google Drive CSV matched pattern: {source_str}")
        latest_source = matches[-1]
    else:
        if not source.exists():
            raise FileNotFoundError(f"Google Drive file not found: {source}")
        latest_source = source

    download_dir.mkdir(parents=True, exist_ok=True)
    if label:
        safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", label).strip("_") or "artifact"
        destination_name = f"{safe_label}__{latest_source.name}"
    else:
        destination_name = latest_source.name
    destination = download_dir / destination_name
    shutil.copy2(latest_source, destination)
    print(f"Downloaded latest Drive artifact: {latest_source} -> {destination}")
    return destination


def load_square_matrix(csv_path: Path) -> pd.DataFrame:
    """Load a labeled square matrix from CSV and enforce validity checks.

    The CSV is expected to have a row index in the first column and matching
    row/column labels describing the same node set.
    """
    df = pd.read_csv(csv_path, index_col=0)

    if df.shape[0] != df.shape[1]:
        if df.shape[1] <= 3:
            raise ValueError(
                f"Matrix at {csv_path} is not square: {df.shape}. "
                "This looks like an edge-list export (deprecated). "
                "Use Graphiko schema adjacency matrix paths, e.g. "
                "/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv"
            )
        raise ValueError(f"Matrix at {csv_path} is not square: {df.shape}")

    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    row_labels = set(df.index)
    col_labels = set(df.columns)
    if row_labels != col_labels:
        raise ValueError(
            f"Row/column label mismatch in {csv_path}. "
            f"Rows={len(row_labels)} unique, Cols={len(col_labels)} unique"
        )

    # Guarantee row/column alignment by ID before any downstream analysis.
    df = df.reindex(index=sorted(df.index), columns=sorted(df.columns))
    df = df.loc[df.index, df.index]
    df = df.apply(pd.to_numeric, errors="raise")
    return df


def align_matrices(a: pd.DataFrame, b: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Restrict both matrices to the shared node intersection in a stable order."""
    common = sorted(set(a.index).intersection(b.index))
    if len(common) < 3:
        raise ValueError("Need at least 3 shared nodes for robust graph similarity stats.")
    return a.loc[common, common], b.loc[common, common]


def minmax_normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Global min-max normalization to [0, 1]; used only for optional visualization transforms."""
    arr = df.to_numpy(dtype=float)
    v_min, v_max = np.nanmin(arr), np.nanmax(arr)
    if np.isclose(v_min, v_max):
        return pd.DataFrame(np.zeros_like(arr), index=df.index, columns=df.columns)
    return pd.DataFrame((arr - v_min) / (v_max - v_min), index=df.index, columns=df.columns)


def distances_to_similarity(df: pd.DataFrame) -> pd.DataFrame:
    """Optional visualization helper: similarity = 1 - minmax(distance)."""
    d = minmax_normalize(df)
    return 1.0 - d


def row_sum_stats(df: pd.DataFrame) -> Dict[str, float]:
    """Diagnostics for row-normalization quality."""
    row_sums = df.sum(axis=1).to_numpy(dtype=float)
    return {
        'row_sum_min': float(np.min(row_sums)),
        'row_sum_max': float(np.max(row_sums)),
        'row_sum_mean': float(np.mean(row_sums)),
        'row_sum_max_abs_deviation_from_1': float(np.max(np.abs(row_sums - 1.0))),
    }


def is_symmetric(df: pd.DataFrame, atol: float = 1e-10) -> bool:
    """Return True when matrix is symmetric within tolerance."""
    arr = df.to_numpy(dtype=float)
    return bool(np.allclose(arr, arr.T, atol=atol, rtol=0.0))


def max_asymmetry(df: pd.DataFrame) -> float:
    """Maximum absolute asymmetry |A - A^T|."""
    arr = df.to_numpy(dtype=float)
    return float(np.max(np.abs(arr - arr.T)))


def off_diagonal_vector(df: pd.DataFrame) -> np.ndarray:
    """Vectorize all directed off-diagonal entries (i, j) where i != j."""
    arr = df.to_numpy(dtype=float)
    mask = ~np.eye(arr.shape[0], dtype=bool)
    return arr[mask]


def jaccard_threshold_sweep(a_vec: np.ndarray, b_vec: np.ndarray, thresholds: np.ndarray) -> Dict[str, float]:
    """Compute Jaccard overlap for multiple directed edge-weight thresholds."""
    out: Dict[str, float] = {}
    for t in thresholds:
        a_edge = a_vec >= t
        b_edge = b_vec >= t
        inter = np.logical_and(a_edge, b_edge).sum()
        union = np.logical_or(a_edge, b_edge).sum()
        out[f"{float(t):.2f}"] = float(inter / union) if union > 0 else float('nan')
    return out


def permutation_distribution(
    a: pd.DataFrame,
    b: pd.DataFrame,
    permutations: int = 2000,
    random_state: int = 42,
) -> Tuple[float, np.ndarray]:
    """Observed Pearson matrix-correlation and permutation null distribution (directed, off-diagonal)."""
    rng = np.random.default_rng(random_state)
    a_vec = off_diagonal_vector(a)
    b_vec = off_diagonal_vector(b)
    obs_r, _ = pearsonr(a_vec, b_vec)

    n = a.shape[0]
    b_arr = b.to_numpy(dtype=float)
    perm_stats = np.empty(permutations, dtype=float)

    for i in range(permutations):
        p = rng.permutation(n)
        b_perm = pd.DataFrame(b_arr[np.ix_(p, p)], index=a.index, columns=a.columns)
        perm_stats[i], _ = pearsonr(a_vec, off_diagonal_vector(b_perm))

    return float(obs_r), perm_stats


def mantel_test(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 42) -> Tuple[float, float]:
    """Mantel-style correlation with two-sided permutation p-value (directed distances)."""
    obs_r, perm_stats = permutation_distribution(a, b, permutations=permutations, random_state=random_state)
    p_value = (np.sum(np.abs(perm_stats) >= np.abs(obs_r)) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def qap_correlation(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 7) -> Tuple[float, float]:
    """QAP-style matrix correlation with one-sided permutation p-value (directed distances)."""
    obs_r, perm_stats = permutation_distribution(a, b, permutations=permutations, random_state=random_state)
    p_value = (np.sum(perm_stats >= obs_r) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def matrix_for_visualization(df: pd.DataFrame, visualization_space: str) -> pd.DataFrame:
    """Render-only matrix transform. Analysis always uses native distance."""
    if visualization_space == 'distance':
        return df.copy()
    if visualization_space == 'similarity':
        return distances_to_similarity(df)
    raise ValueError("VISUALIZATION_SPACE must be 'distance' or 'similarity'.")


def compute_similarity_metrics(
    embeddings_matrix: pd.DataFrame,
    subscriptions_matrix: pd.DataFrame,
    comparison_mode: str,
    analysis_space: str,
    edge_thresholds: np.ndarray,
    permutations: int,
) -> SimilarityResult:
    """Compute a suite of directed graph similarity metrics in native distance space."""
    emb_vec = off_diagonal_vector(embeddings_matrix)
    sub_vec = off_diagonal_vector(subscriptions_matrix)

    pear_r, pear_p = pearsonr(emb_vec, sub_vec)
    spr_rho, spr_p = spearmanr(emb_vec, sub_vec)
    cos_sim = 1.0 - cosine(emb_vec, sub_vec)

    rmse = float(np.sqrt(np.mean((emb_vec - sub_vec) ** 2)))
    mae = float(np.mean(np.abs(emb_vec - sub_vec)))

    edge_jaccard_by_threshold = jaccard_threshold_sweep(emb_vec, sub_vec, edge_thresholds)

    mantel_r, mantel_p = mantel_test(embeddings_matrix, subscriptions_matrix, permutations)
    qap_r, qap_p = qap_correlation(embeddings_matrix, subscriptions_matrix, permutations)

    return SimilarityResult(
        comparison_mode=comparison_mode,
        analysis_space=analysis_space,
        n_nodes=embeddings_matrix.shape[0],
        n_pairs=emb_vec.size,
        pearson_r=float(pear_r),
        pearson_p=float(pear_p),
        spearman_rho=float(spr_rho),
        spearman_p=float(spr_p),
        cosine_similarity=float(cos_sim),
        rmse=rmse,
        mae=mae,
        edge_jaccard_by_threshold=edge_jaccard_by_threshold,
        mantel_r=mantel_r,
        mantel_p=mantel_p,
        qap_r=qap_r,
        qap_p=qap_p,
    )


def save_visualizations(
    analysis_embeddings: pd.DataFrame,
    analysis_subscriptions: pd.DataFrame,
    plot_embeddings: pd.DataFrame,
    plot_subscriptions: pd.DataFrame,
    output_dir: Path,
    permutations: int = 2000,
    visualization_space: str = 'distance',
) -> Dict[str, str]:
    """Render and save graph comparison figures. Uses native distances for stats; transform is render-only."""
    output_dir.mkdir(parents=True, exist_ok=True)
    out: Dict[str, str] = {}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    sns.heatmap(plot_embeddings, cmap="viridis", ax=axes[0], cbar=True)
    axes[0].set_title(f"Embeddings Graph ({visualization_space}, directed)")
    sns.heatmap(plot_subscriptions, cmap="viridis", ax=axes[1], cbar=True)
    axes[1].set_title(f"Subscriptions Graph ({visualization_space}, directed)")
    p1 = output_dir / "heatmaps_side_by_side.png"
    fig.savefig(p1, dpi=220)
    plt.close(fig)
    out["heatmaps"] = str(p1)

    diff = analysis_embeddings - analysis_subscriptions
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.heatmap(diff, cmap="coolwarm", center=0.0, ax=ax)
    ax.set_title("Directed Distance Difference Matrix (Embeddings - Subscriptions)")
    p2 = output_dir / "difference_heatmap.png"
    fig.savefig(p2, dpi=220)
    plt.close(fig)
    out["difference_heatmap"] = str(p2)

    emb_vec = off_diagonal_vector(analysis_embeddings)
    sub_vec = off_diagonal_vector(analysis_subscriptions)
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.regplot(x=emb_vec, y=sub_vec, scatter_kws={"alpha": 0.45, "s": 18}, ax=ax)
    ax.set_xlabel("Embeddings directed edge distance (ordered pair weight)")
    ax.set_ylabel("Subscriptions directed edge distance (ordered pair weight)")
    ax.set_title("Directed ordered-pair relationship between distance matrices")
    p3 = output_dir / "edge_weight_scatter.png"
    fig.savefig(p3, dpi=220)
    plt.close(fig)
    out["scatter"] = str(p3)

    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.histplot(emb_vec, bins=40, color="#1f77b4", alpha=0.45, stat="density", label="Embeddings", ax=ax)
    sns.histplot(sub_vec, bins=40, color="#ff7f0e", alpha=0.45, stat="density", label="Subscriptions", ax=ax)
    ax.set_xlabel("Directed edge distance (ordered pair weight)")
    ax.set_ylabel("Density")
    ax.set_title("Directed edge-distance distribution overlap")
    ax.legend()
    p4 = output_dir / "edge_weight_distributions.png"
    fig.savefig(p4, dpi=220)
    plt.close(fig)
    out["distribution_overlap"] = str(p4)

    mantel_obs, mantel_perm = permutation_distribution(analysis_embeddings, analysis_subscriptions, permutations=permutations, random_state=42)
    qap_obs, qap_perm = permutation_distribution(analysis_embeddings, analysis_subscriptions, permutations=permutations, random_state=7)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    sns.histplot(mantel_perm, bins=40, stat="density", color="#2ca02c", alpha=0.75, ax=axes[0])
    axes[0].axvline(mantel_obs, color="black", linestyle="--", linewidth=2, label=f"Observed r={mantel_obs:.3f}")
    axes[0].set_title("Mantel-style directed permutation null distribution")
    axes[0].set_xlabel("Permutation correlation (off-diagonal directed pairs)")
    axes[0].legend()

    sns.histplot(qap_perm, bins=40, stat="density", color="#9467bd", alpha=0.75, ax=axes[1])
    axes[1].axvline(qap_obs, color="black", linestyle="--", linewidth=2, label=f"Observed r={qap_obs:.3f}")
    axes[1].set_title("QAP-style directed permutation null distribution")
    axes[1].set_xlabel("Permutation correlation (off-diagonal directed pairs)")
    axes[1].legend()

    p5 = output_dir / "permutation_null_distributions.png"
    fig.savefig(p5, dpi=220)
    plt.close(fig)
    out["permutation_nulls"] = str(p5)

    return out



## Configure your run

Update the two Google Drive input paths (or glob patterns) below.

- The notebook copies the **latest Google Drive artifact** for each input into local runtime storage before analysis.
- If your files represent **distance matrices**, keep `DISTANCE_TO_SIMILARITY = True`.
- If they are already **similarities**, set it to `False`.


In [11]:
# ----------------------------
# User configuration
# ----------------------------
EMBEDDINGS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv')
SUBSCRIPTIONS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv')
LOCAL_DOWNLOAD_DIR = Path('inputs/downloaded_from_drive')
OUTPUT_DIR = Path('outputs/graph_similarity')

# Main analysis default: compare native row-normalized directed distances.
COMPARISON_MODE = 'directed_row_normalized_distance'
ANALYSIS_SPACE = 'native_distance'

# Visualization-only transform selector for heatmaps.
VISUALIZATION_SPACE = 'distance'  # 'distance' or 'similarity'

# Jaccard overlap is reported as a threshold sweep in directed distance space.
EDGE_THRESHOLDS = np.array([0.10, 0.20, 0.30, 0.40, 0.50])
PERMUTATIONS = 2000



In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


> **Note:** The main analysis compares native row-normalized directed distance matrices. Any similarity transformation is visualization-only unless explicitly enabled.


In [14]:
# ----------------------------
# Execute analysis
# ----------------------------
EMBEDDINGS_CSV = resolve_latest_drive_csv(EMBEDDINGS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR, label='embeddings')
SUBSCRIPTIONS_CSV = resolve_latest_drive_csv(SUBSCRIPTIONS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR, label='subscriptions')

emb_raw = load_square_matrix(EMBEDDINGS_CSV)
sub_raw = load_square_matrix(SUBSCRIPTIONS_CSV)

emb_aligned, sub_aligned = align_matrices(emb_raw, sub_raw)

# Primary analysis path: do not re-normalize or invert.
if ANALYSIS_SPACE != 'native_distance':
    raise ValueError("ANALYSIS_SPACE must remain 'native_distance' for this notebook's default analysis path.")

emb_analysis = emb_aligned.copy()
sub_analysis = sub_aligned.copy()

# Visualization-only rendering matrices.
emb_plot = matrix_for_visualization(emb_analysis, VISUALIZATION_SPACE)
sub_plot = matrix_for_visualization(sub_analysis, VISUALIZATION_SPACE)

result = compute_similarity_metrics(
    emb_analysis,
    sub_analysis,
    comparison_mode=COMPARISON_MODE,
    analysis_space=ANALYSIS_SPACE,
    edge_thresholds=EDGE_THRESHOLDS,
    permutations=PERMUTATIONS,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics = asdict(result)

metrics_json = OUTPUT_DIR / 'similarity_metrics.json'
metrics_csv = OUTPUT_DIR / 'similarity_metrics.csv'
summary_json = OUTPUT_DIR / 'run_summary.json'

with metrics_json.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame([metrics]).to_csv(metrics_csv, index=False)
plots = save_visualizations(
    emb_analysis,
    sub_analysis,
    emb_plot,
    sub_plot,
    OUTPUT_DIR,
    permutations=PERMUTATIONS,
    visualization_space=VISUALIZATION_SPACE,
)

summary = {
    'inputs': {
        'embeddings_drive_source': str(EMBEDDINGS_DRIVE_SOURCE),
        'subscriptions_drive_source': str(SUBSCRIPTIONS_DRIVE_SOURCE),
        'embeddings_downloaded_csv': str(EMBEDDINGS_CSV),
        'subscriptions_downloaded_csv': str(SUBSCRIPTIONS_CSV),
        'comparison_mode': COMPARISON_MODE,
        'analysis_space': ANALYSIS_SPACE,
        'visualization_space': VISUALIZATION_SPACE,
        'edge_thresholds': [float(x) for x in EDGE_THRESHOLDS],
    },
    'matrix_diagnostics': {
        'embeddings': {
            'is_symmetric': is_symmetric(emb_analysis),
            'max_asymmetry': max_asymmetry(emb_analysis),
            **row_sum_stats(emb_analysis),
        },
        'subscriptions': {
            'is_symmetric': is_symmetric(sub_analysis),
            'max_asymmetry': max_asymmetry(sub_analysis),
            **row_sum_stats(sub_analysis),
        },
    },
    'metrics_json': str(metrics_json),
    'metrics_csv': str(metrics_csv),
    'plots': plots,
}

with summary_json.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Graph similarity analysis complete.')
print(json.dumps(summary, indent=2))
pd.DataFrame([metrics]).T.rename(columns={0: 'value'})

Downloaded latest Drive artifact: /content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv -> inputs/downloaded_from_drive/embeddings__adjacency_matrix.csv
Downloaded latest Drive artifact: /content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv -> inputs/downloaded_from_drive/subscriptions__adjacency_matrix.csv
Graph similarity analysis complete.
{
  "inputs": {
    "embeddings_drive_source": "/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv",
    "subscriptions_drive_source": "/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv",
    "embeddings_downloaded_csv": "inputs/downloaded_from_drive/embeddings__adjacency_matrix.csv",
    "subscriptions_downloaded_csv": "inputs/downloaded_from_drive/subscriptions__adjacency_matrix.csv",
    "comparison_mode": "directed_row_normalized_distance",
    "analysis_space": "native_distance"

,value
comparison_mode,directed_row_normalized_distance
analysis_space,native_distance
n_nodes,27
n_pairs,702
pearson_r,0.119109
pearson_p,0.00157
spearman_rho,0.156166
spearman_p,0.000032
cosine_similarity,0.894906
rmse,0.019214
